In [2]:
# Note sulla versione: questo notepad è stato fatto con un ambiente conda con python 3.6 per la compatibilità con anc2vec.
# Per riprodurre: https://github.com/sinc-lab/anc2vec
# 
# Questo notepad è dedicato alla fase di data pre-processing antecedente all'elaborazione in GNN
# Passaggi:
#   1. Generazione matrice di adiacenza (o altra rappresentazione compatibile con GNN)
#   2. Generazione matrice delle feature
#   2.1     Encoding delle GO

In [3]:
# Qui verranno ottenuti gli embedding per le GO presenti nel grafo da analizzare
import pandas as pd
import numpy as np
import anc2vec

embeddings = anc2vec.get_embeddings()

In [4]:
import torch

# Apro i file dei nodi e degli archi
nodes_file = '../files/networks/union_nodes.csv'
nodes_df = pd.read_csv(nodes_file)[['name', 'Gene Ontology IDs']].dropna(subset=['Gene Ontology IDs']) # N.B. c'è una proteina senza GO
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}

edges_file = '../files/networks/union_edges.csv'
raw_edges_df = pd.read_csv(edges_file)['name']

# Ottengo una lista di go e li associo agli embedding
unique_go_ids = set(
    id.strip() for ids in nodes_df['Gene Ontology IDs'].dropna().str.split(';') for id in ids
)
unique_go_ids_list = list(unique_go_ids)

# NOTA: questa è una prova, vengono saltati i termini che il modello non conosce (andrebbe ri addestrato)
go_to_embedding = {go:embeddings[go] for go in unique_go_ids if go in embeddings}

def merge_embeddings(go_list):
    embeddings = [go_to_embedding[g.strip()] for g in go_list if g in go_to_embedding]
    return np.mean(embeddings, axis=0).tolist()

# Sostituisco i nodi con degli id
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}
edges_df = raw_edges_df.str.extract(r'(\S+) \(interacts with\) (\S+)')
edges_df.columns = ['node1', 'node2']

edges_df['node1'] = edges_df['node1'].map(node_ids)
edges_df['node2'] = edges_df['node2'].map(node_ids)

edges_df = edges_df[~edges_df["node2"].isna()] # Una proteina non è annotata, da aggiustare dopo aver fatto il prototipo

# Definisco il tensore degli archi (ogni riga indica un arco)
edge_index = torch.tensor(edges_df[["node1", "node2"]].values, dtype=torch.long).t().contiguous()

# Sostituisco gli id dei nodi con quelli generati prima
nodes_df['name'] = nodes_df['name'].map(node_ids)

# Aggrega tutte le go di ogni record
nodes_df['Gene Ontology IDs'] = nodes_df['Gene Ontology IDs'].dropna().apply(
    lambda x: merge_embeddings([go.strip() for go in x.split(';')])
)
features = []
for c in nodes_df['Gene Ontology IDs'].dropna():
    features.append(c)

# Crea il vettore di feature
x = torch.tensor(features, dtype=torch.float)  # Esclude "Entry Name"

In [5]:
# Qui serializzo le strutture necessarie al training della GNN
# Questo perché questo notepad è python3.6 per compatibilità con anc2vec, PyG invece vuole python >= 3.9

torch.save({
    'edge_index': edge_index,
    'node_features': x,
    'node_ids': node_ids
}, './model_output/pre_processed_data.pt')

In [ ]:
# QUESTO PAD È IDENTICO A QUELLO DI SOPRA CON L'UNICA DIFFERENZA CHE FILTRA LE GO PER QUELLE TROVATE NEL DATASET DI CTD

import pickle

with open('CTD_gos.pkl', 'rb') as f:
    ctd_go_set = pickle.load(f)

import torch

# Apro i file dei nodi e degli archi
nodes_file = '../files/networks/union_nodes.csv'
nodes_df = pd.read_csv(nodes_file)[['name', 'Gene Ontology IDs']].dropna(subset=['Gene Ontology IDs']) # N.B. c'è una proteina senza GO
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}

edges_file = '../files/networks/union_edges.csv'
raw_edges_df = pd.read_csv(edges_file)['name']

# Ottengo una lista di go e li associo agli embedding
unique_go_ids = set(
    id.strip() for ids in nodes_df['Gene Ontology IDs'].dropna().str.split(';') for id in ids
)

print('Numero di GO prima del filtro: ', len(unique_go_ids))
unique_go_ids = unique_go_ids & ctd_go_set
print('Numero di GO dopo il filtro: ', len(unique_go_ids))

# NOTA: questa è una prova, vengono saltati i termini che il modello non conosce (andrebbe ri addestrato)
go_to_embedding = {go:embeddings[go] for go in unique_go_ids if go in embeddings}

def merge_embeddings(go_list):
    embeddings = [go_to_embedding[g.strip()] for g in go_list if g in go_to_embedding]
    return np.mean(embeddings, axis=0).tolist() if embeddings else None

# Sostituisco i nodi con degli id
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}
edges_df = raw_edges_df.str.extract(r'(\S+) \(interacts with\) (\S+)')
edges_df.columns = ['node1', 'node2']

edges_df['node1'] = edges_df['node1'].map(node_ids)
edges_df['node2'] = edges_df['node2'].map(node_ids)

print("Numero di node1 Nan in edges_df: ", len(edges_df[edges_df["node1"].isna()]))
print("Numero di node2 Nan in edges_df: ", len(edges_df[edges_df["node2"].isna()]))
edges_df = edges_df[~edges_df["node2"].isna()] # Alcune proteine non sono annotate, quindi non hanno un id

# Definisco il tensore degli archi (ogni riga indica un arco)
edge_index = torch.tensor(edges_df[["node1", "node2"]].values, dtype=torch.long).t().contiguous()

# Sostituisco gli id dei nodi con quelli generati prima
nodes_df['name'] = nodes_df['name'].map(node_ids)

print("Presenza di Nan in nodes_df[name]", nodes_df['name'].isna().any())
print("Presenza di NaN in colonna Gene Ontology IDs: ", nodes_df['Gene Ontology IDs'].isna().any())

# Aggrega tutte le go di ogni record
nodes_df['Gene Ontology IDs'] = nodes_df['Gene Ontology IDs'].dropna().apply(
    lambda x: merge_embeddings([go.strip() for go in x.split(';')])
)

# Elimina i record con le go nulle
print("Record di nodes_df con go non riconosciute dal modello: ", len(nodes_df[nodes_df['Gene Ontology IDs'].isna()]))

print("Nodi con go nan", nodes_df[nodes_df['Gene Ontology IDs'].isna()])

features = []
for c in nodes_df['Gene Ontology IDs'].dropna():
    features.append(c)

print("Len features: ", len(features))

# Crea il vettore di feature
x = torch.tensor(features, dtype=torch.float)  # Esclude "Entry Name"
print("feature shape: ", x.shape[0])
print("Massimo ID presente in edge_index:", edge_index.max().item())
print("Minimo ID presente in edge_index:", edge_index.min().item())

# Ricreiamo node_ids solo con i nodi ancora validi
node_ids = {node: i for i, node in enumerate(nodes_df.dropna(subset=['Gene Ontology IDs'])['name'].tolist())}
# Aggiorniamo edges_df con i nuovi ID
edges_df['node1'] = edges_df['node1'].map(node_ids)
edges_df['node2'] = edges_df['node2'].map(node_ids)

# Rimuoviamo gli archi che contengono NaN dopo la mappatura
edges_df = edges_df.dropna().astype(int)

# Ricostruiamo edge_index
edge_index = torch.tensor(edges_df[["node1", "node2"]].values, dtype=torch.long).t().contiguous()
print("Shape di x:", x.shape[0])
print("Massimo ID in edge_index:", edge_index.max().item())
print("Minimo ID in edge_index:", edge_index.min().item())

assert edge_index.max().item() < x.shape[0], "Errore: edge_index contiene ID di nodi non presenti in x!"



Numero di GO prima del filtro:  2477
Numero di GO dopo il filtro:  1170
Numero di node1 Nan in edges_df:  0
Numero di node2 Nan in edges_df:  3
Presenza di Nan in nodes_df[name] False
Presenza di NaN in colonna Gene Ontology IDs:  False
Record di nodes_df con go non riconosciute dal modello:  6
Nodi con go nan      name Gene Ontology IDs
25     25              None
40     40              None
41     41              None
85     85              None
151   151              None
172   171              None
Len features:  172
feature shape:  172
Massimo ID presente in edge_index: 177
Minimo ID presente in edge_index: 0
Shape di x: 172
Massimo ID in edge_index: 171
Minimo ID in edge_index: 0


In [26]:
torch.save({
    'edge_index': edge_index,
    'node_features': x,
    'node_ids': node_ids
}, './model_output/pre_processed_data_ctd.pt')